# Làm sạch dữ liệu `url_dataset.csv`

Bộ dữ liệu gồm các URL kèm nhãn `type`: `legitimate` (trang bình thường) hoặc `phishing` (trang lừa đảo). Notebook này đọc file gốc, tìm lỗi, sửa hoặc loại bỏ, rồi lưu file sạch.

**Cách chạy:** để notebook cạnh file `url_dataset.csv`, chọn *Kernel → Restart & Run All* (cần Python 3 và pandas từ 1.1 trở lên). Đường dẫn và các lựa chọn nằm ở ô cấu hình ngay bên dưới.

| Bước | Việc làm |
|---|---|
| 1 | Đọc dữ liệu |
| 2 | Khảo sát tổng quan: kích thước, giá trị thiếu, dòng trùng, nhãn, độ dài |
| 3 | Lập bảng đếm lỗi trong nội dung URL |
| 4 | **Sửa** những lỗi có cách sửa rõ ràng (giữ lại dòng) |
| 5 | **Loại** những dòng không dùng được: URL bị cắt cụt, host sai, trùng lặp, nhãn mâu thuẫn |
| 6 | Kiểm tra lại sau khi làm sạch |
| 7 | Lưu file sạch và hai file nhật ký (dòng nào bị sửa, dòng nào bị loại, vì sao) |
| 8 | Lưu ý khi dùng dữ liệu này để huấn luyện mô hình |

Nguyên tắc: **không đụng vào nhãn, không đoán nội dung URL**. Lỗi nào có cách sửa chắc chắn thì sửa. Lỗi nào không chắc thì loại dòng đó và ghi lý do vào nhật ký, để mọi thay đổi đều truy ngược được.

In [ ]:
import html
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

# ---- Cấu hình ----
INPUT_PATH = Path("url_dataset.csv")              # file gốc, để cạnh notebook
OUTPUT_PATH = Path("url_dataset_clean.csv")       # file sạch
CHANGES_PATH = Path("url_dataset_changes.csv")    # nhật ký: URL nào đã bị sửa
DROPPED_PATH = Path("url_dataset_dropped.csv")    # nhật ký: dòng nào bị loại, vì sao

# True  = loại các URL bị cắt cụt (kết thúc bằng "...")  -> xem mục 5.1
# False = chỉ bỏ dấu "..." rồi giữ dòng lại
DROP_TRUNCATED = True

LABELS = {"legitimate", "phishing"}

## 1. Đọc dữ liệu

Đọc mọi cột dưới dạng chuỗi (`dtype=object`) và tắt `keep_default_na`. Mặc định pandas tự đổi các chuỗi như `"NA"`, `"null"`, `"nan"` thành giá trị thiếu. Với dữ liệu URL, ta muốn giữ nguyên văn để tự kiểm soát.

`row_id` là số thứ tự dòng trong file gốc (bắt đầu từ 0). Ta giữ nó làm index để dòng nào bị sửa hay bị loại cũng truy ngược được. Số dòng trong file CSV = `row_id + 2` (vì có dòng tiêu đề).

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Không thấy {INPUT_PATH.resolve()} - hãy để file CSV cạnh notebook.")

raw = pd.read_csv(INPUT_PATH, dtype=object, keep_default_na=False, na_values=[])
raw.index.name = "row_id"
print(f"{len(raw):,} dòng x {raw.shape[1]} cột")
raw.head()

450,176 dòng x 2 cột


,url,type
row_id,,
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate


## 2. Khảo sát nhanh

Trước khi sửa, xem tổng quan: kiểu dữ liệu, giá trị thiếu, dòng trùng, phân bố nhãn, độ dài URL.

In [ ]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450176 entries, 0 to 450175
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     450176 non-null  object
 1   type    450176 non-null  object
dtypes: object(2)
memory usage: 6.9+ MB


In [ ]:
print("Số ô rỗng (kể cả chỉ có khoảng trắng):")
print(raw.apply(lambda c: c.str.strip() == "").sum().to_string())
print()
print("Dòng trùng hoàn toàn (url + type):", raw.duplicated().sum())
print("URL trùng nhau (bất kể nhãn):     ", raw["url"].duplicated().sum())
print()

counts = raw["type"].value_counts()
display(pd.DataFrame({"số dòng": counts, "tỉ lệ": (counts / len(raw)).round(4)}))

Số ô rỗng (kể cả chỉ có khoảng trắng):
url     0
type    0

Dòng trùng hoàn toàn (url + type): 0
URL trùng nhau (bất kể nhãn):      0



,số dòng,tỉ lệ
type,,
legitimate,345738,0.768
phishing,104438,0.232


In [ ]:
length = raw["url"].str.len()
display(length.describe().round(1).to_frame("độ dài URL"))

longest = raw.loc[length.nlargest(5).index].assign(length=length)
longest["url"] = longest["url"].str.slice(0, 90) + "..."
display(longest[["length", "type", "url"]])

,độ dài URL
count,450176.0
mean,60.2
std,37.6
min,8.0
25%,40.0
50%,52.0
75%,71.0
max,2314.0


,length,type,url
row_id,,,
355326,2314,phishing,http://cadastroclienteseguro.com/ambienteseguranca/login.do2562.html?78dw8afuvrmtzpefdmhc3...
440342,1970,phishing,http://samchak.com/?ga=uhhbsxXeH/6EMcmFRUBZcNYHV94RtYiSCnLIuAAH932qwW//RqUoFq0X5bveSKSyusC...
422612,1919,phishing,http://www.transaz.com/sql.php/PCFET0NUWVBFIEhUTUwgUFVCTElDICItLy9XM0MvL0RURCBIVE1MIDQuMDE...
389379,1673,phishing,http://ib.adnxs.com/tt?id=2614296&referrer=http%3A%2F%2Falert.scansafe.net%2Falert%2Fproce...
389304,1499,phishing,http://ams1.ib.adnxs.com/if?enc=MzMzMzMzwz-5SQwCK4e-P8HKoUW28-E_uUkMAiuHvj8zMzMzMzPDPyA2hb...


**Nhận xét.** Không có ô rỗng, không có dòng trùng hoàn toàn, cột `type` chỉ có hai giá trị hợp lệ và không dính khoảng trắng thừa, nên cột này không cần sửa. Nhãn lệch về `legitimate` (tỉ lệ xem ở bảng trên).

Độ dài URL lệch phải mạnh (trung vị chỉ khoảng 50 ký tự trong khi dài nhất hơn 2.000). Những URL rất dài này là hợp lệ (chuỗi mã hóa nằm trong phần query) nên **giữ lại**, không coi là outlier để xóa.

Vấn đề thật nằm ở *nội dung* URL, xem mục 3.

## 3. Bảng lỗi trong nội dung URL

"Không thiếu, không trùng" mới là sạch ở mức bảng. Ta viết một nhóm hàm kiểm tra (dùng regex) để đếm số dòng dính từng loại lỗi. Cùng bộ hàm này sẽ chạy lại ở mục 6 để chứng minh lỗi đã hết.

In [ ]:
URL_PARTS = re.compile(r"^([A-Za-z][A-Za-z0-9+.\-]*)://([^/?#]*)(.*)$", re.S)   # scheme :// authority rest
ENTITY = re.compile(r"&(?:amp|lt|gt|quot|apos|#\d+|#[xX][0-9a-fA-F]+);")          # &amp; &lt; &#39; ...
# Ký tự điều khiển và ký tự vô hình: mã 0-31 và 127, dấu nối mềm, zero-width, ký tự đổi hướng chữ, BOM...
INVISIBLE_RANGES = [(0x00, 0x1F), (0x7F, 0x7F), (0xAD, 0xAD), (0x200B, 0x200F),
                    (0x202A, 0x202E), (0x2060, 0x2064), (0xFEFF, 0xFEFF)]
INVISIBLE = re.compile("[" + "".join(f"{re.escape(chr(a))}-{re.escape(chr(b))}" for a, b in INVISIBLE_RANGES) + "]")


def split_url(s):
    """Tách mỗi URL thành scheme / authority (user@host:port) / phần còn lại (path, query...)."""
    parts = s.str.extract(URL_PARTS)
    parts.columns = ["scheme", "authority", "rest"]
    return parts


def get_host(s):
    """Lấy host: bỏ 'user:pass@', bỏ ':port', bỏ dấu chấm cuối (tên miền tuyệt đối)."""
    authority = split_url(s)["authority"].fillna("")
    host = authority.str.rsplit("@", n=1).str[-1]
    return host.str.replace(r":\d*$", "", regex=True).str.replace(r"\.$", "", regex=True).rename("host")


def valid_host(s):
    """Host hợp lệ: các nhãn chữ/số/gạch nối/gạch dưới, nối bằng dấu chấm, có ít nhất một dấu chấm."""
    has_scheme = split_url(s)["scheme"].notna()
    ok = get_host(s).str.fullmatch(r"(?:[\w-]+\.)+[\w-]+").astype(bool)
    return has_scheme & ok


def has_upper_scheme_host(s):
    """True nếu scheme hoặc host có chữ hoa (phần user:pass@ không tính)."""
    parts = split_url(s)
    scheme = parts["scheme"].fillna("")
    hostport = parts["authority"].fillna("").str.rsplit("@", n=1).str[-1]
    return (scheme != scheme.str.lower()) | (hostport != hostport.str.lower())


CHECKS = {
    "HTML entity (&amp; &lt; &#39; ...)":     lambda s: s.str.contains(ENTITY),
    "Có khoảng trắng":                        lambda s: s.str.contains(r"\s"),
    "Có ký tự điều khiển / vô hình":          lambda s: s.str.contains(INVISIBLE),
    "Dạng làm mờ [.]":                        lambda s: s.str.contains(r"\[\.\]"),
    "Scheme gõ sai (httpss, htpp, hxxp)":     lambda s: s.str.contains(r"^(?:httpss|htpp|hxxps?)://"),
    "Scheme bị lặp (http://http://...)":      lambda s: s.str.contains(r"^[A-Za-z]+://(?:https?|hxxps?|htpp|ftp)\s*:\s*/"),
    "Scheme/host viết hoa":                   has_upper_scheme_host,
    "URL bị cắt cụt (kết thúc bằng '...')":   lambda s: s.str.endswith("..."),
    "Host không hợp lệ":                      lambda s: ~valid_host(s),
}


def quality_report(s):
    return pd.Series({name: int(check(s).sum()) for name, check in CHECKS.items()})


issues = pd.DataFrame({"số dòng": quality_report(raw["url"])})
issues["% tổng"] = (issues["số dòng"] / len(raw) * 100).round(2)
issues

,số dòng,% tổng
HTML entity (&amp; &lt; &#39; ...),5524,1.23
Có khoảng trắng,521,0.12
Có ký tự điều khiển / vô hình,5,0.00
Dạng làm mờ [.],1,0.00
"Scheme gõ sai (httpss, htpp, hxxp)",35,0.01
Scheme bị lặp (http://http://...),120,0.03
Scheme/host viết hoa,160,0.04
URL bị cắt cụt (kết thúc bằng '...'),2555,0.57
Host không hợp lệ,658,0.15


Ngoài ra có vài đặc điểm **chỉ ghi nhận, không sửa** vì chúng là thông tin thật của URL chứ không phải lỗi:

In [ ]:
NOTES = {
    "Có ký tự ngoài ASCII (é, ’, –...)":         lambda s: s.str.contains(r"[^\x00-\x7f]"),
    "Nghi lỗi mã hóa chữ (Ã©, â€“...)":          lambda s: s.str.contains(r"Ã[\x80-\xbf]|Â[\x80-\xbf]|â€"),
    "Host là địa chỉ IP":                        lambda s: get_host(s).str.fullmatch(r"\d{1,3}(?:\.\d{1,3}){3}").astype(bool),
    "URL dài hơn 1.000 ký tự":                   lambda s: s.str.len() > 1000,
}
pd.DataFrame({"số dòng": {name: int(f(raw["url"]).sum()) for name, f in NOTES.items()}})

,số dòng
"Có ký tự ngoài ASCII (é, ’, –...)",310
"Nghi lỗi mã hóa chữ (Ã©, â€“...)",8
Host là địa chỉ IP,3954
URL dài hơn 1.000 ký tự,24


Chữ có dấu trong URL là hợp lệ (tên bài viết, tên người). Số ít dòng lỗi mã hóa chữ chỉ nằm ở phần path, không ảnh hưởng tên miền, nên để nguyên. Host là IP là một đặc trưng hữu ích để phát hiện phishing, giữ nguyên.

## 4. Sửa lỗi (giữ lại dòng)

Mỗi quy tắc là một hàm nhận cột `url` và trả về cột `url` đã sửa. Hàm `apply_rule` áp dụng quy tắc, đếm số dòng thay đổi, ghi lại quy tắc nào đã đụng vào dòng nào (để làm nhật ký ở mục 7) và in vài ví dụ trước/sau.

Thứ tự quy tắc có ý nghĩa: giải mã entity trước (vì `&#32;` giải ra khoảng trắng), xử lý khoảng trắng trước khi sửa scheme (vì `http :// abc.com` phải được nối lại trước), và hạ chữ thường host sau cùng.

In [ ]:
df = raw.copy()
applied = defaultdict(list)    # row_id -> các quy tắc đã đụng vào dòng đó


def diff_window(a, b, width=40):
    """Cắt đoạn quanh chỗ khác nhau đầu tiên của hai chuỗi. Dùng repr để thấy cả ký tự vô hình (\\x13, \\xa0...)."""
    i = next((k for k, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
    lo, hi = max(0, i - width), i + width

    def cut(t):
        return ("…" if lo > 0 else "") + repr(t[lo:hi])[1:-1] + ("…" if hi < len(t) else "")

    return cut(a), cut(b)


def apply_rule(name, func, n_examples=3):
    before = df["url"]
    after = func(before)
    changed = before != after
    for rid in before.index[changed]:
        applied[rid].append(name)
    df["url"] = after
    print(f"{name}: {int(changed.sum()):,} dòng được sửa")
    examples = [diff_window(a, b) for a, b in zip(before[changed].head(n_examples), after[changed].head(n_examples))]
    if examples:
        display(pd.DataFrame(examples, columns=["trước", "sau"]))

### 4.1 Giải mã HTML entity

Nhiều URL chứa `&amp;` thay vì `&` (thường là dấu vết của việc chép từ trang HTML/XML). Hơn 98% số dòng này là phishing, nên nếu để nguyên, mô hình có thể học "có `&amp;` thì là phishing". Nhiều khả năng đó là dấu vết của cách thu thập dữ liệu chứ không phải đặc điểm của trang web.

Chỉ giải mã các entity dạng đầy đủ có dấu `;`. Không dùng `html.unescape` cho cả chuỗi vì nó biến `&copy=1` thành `©=1`. Một số URL bị escape nhiều lần (`&amp;amp;...`) nên ta lặp đến khi hết.

In [ ]:
def decode_entities(s):
    s = s.copy()
    todo = s.str.contains(ENTITY)
    while todo.any():    # lặp vì có URL bị escape nhiều lần; mỗi vòng chuỗi ngắn đi nên chắc chắn dừng
        s[todo] = s[todo].str.replace(ENTITY, lambda m: html.unescape(m.group(0)), regex=True)
        todo = todo & s.str.contains(ENTITY)
    return s


apply_rule("Giải mã HTML entity", decode_entities)

Giải mã HTML entity: 5,524 dòng được sửa


,trước,sau
0,…les-reid.html?showComment=1320272226102&amp;_escaped_fragment_=/2011/11/advice-f…,…les-reid.html?showComment=1320272226102&_escaped_fragment_=/2011/11/advice-from-…
1,….czq0tatv.strategia-tactica.ru/?n=34251&amp;tid=56502,….czq0tatv.strategia-tactica.ru/?n=34251&tid=56502
2,…ons.ca/ElectoralDistricts_e.aspx?type=1&amp;criteria=K0k3h0,…ons.ca/ElectoralDistricts_e.aspx?type=1&criteria=K0k3h0


### 4.2 Khoảng trắng

Khoảng trắng (gồm cả ký tự Unicode như `U+00A0`, `U+200A`) không được có trong URL. Có hai kiểu, xử lý khác nhau:

- **Sát dấu phân cách** (`http://abc.com /`, `www . site . com / a / b`, `sms-trust. com`): lỗi định dạng khi chép dữ liệu, bỏ hẳn khoảng trắng.
- **Còn lại**, nằm giữa tên thư mục hoặc tên file (`/Welcome to AOL.htm`): đổi thành `%20`, đúng chuẩn URL.

In [ ]:
DELIMS = r"[./?&=:\-_#@%,;]"

apply_rule("Xóa khoảng trắng thừa (đầu/cuối, sát dấu phân cách)",
           lambda s: s.str.strip().str.replace(rf"\s*({DELIMS})\s*", r"\1", regex=True))

apply_rule("Khoảng trắng còn lại -> %20",
           lambda s: s.str.replace(r"\s+", "%20", regex=True))

Xóa khoảng trắng thừa (đầu/cuối, sát dấu phân cách): 453 dòng được sửa


,trước,sau
0,…o/alibaba/index.php?email=info@ril.com&. rand=13vqcr8b0gud&lc=1033&id=64855&mkt=…,…o/alibaba/index.php?email=info@ril.com&.rand=13vqcr8b0gud&lc=1033&id=64855&mkt=e…
1,…9642&fid.1&fav.1&email=abuse@laugier.com &.rand=13InboxLight.aspx?n=1774256418&f…,…9642&fid.1&fav.1&email=abuse@laugier.com&.rand=13InboxLight.aspx?n=1774256418&fi…
2,…/ib.adnxs.com/tt?id=2469373&size=728x90& referrer=[HEIAS_REFERRER_ENC],…/ib.adnxs.com/tt?id=2469373&size=728x90&referrer=[HEIAS_REFERRER_ENC]


Khoảng trắng còn lại -> %20: 73 dòng được sửa


,trước,sau
0,http://tzczz.gscvz.online/Welcome to AOL.htm,http://tzczz.gscvz.online/Welcome%20to%20AOL.htm
1,http://willcanvelt.com/Inversiones Will Canvelt/contacto/css/en/B...,http://willcanvelt.com/Inversiones%20Will%20Canvelt/contacto/css/en/B...
2,…/directxex.com/uploads/815597715.bot.exe Trojan.Agent,…/directxex.com/uploads/815597715.bot.exe%20Trojan.Agent


### 4.3 Ký tự điều khiển và ký tự vô hình

Ký tự như `\x13` hay dấu nối mềm `U+00AD` không hiển thị nhưng làm hỏng host hoặc query. Xóa.

In [ ]:
apply_rule("Xóa ký tự điều khiển / vô hình", lambda s: s.str.replace(INVISIBLE, "", regex=True), n_examples=5)

Xóa ký tự điều khiển / vô hình: 5 dòng được sửa


,trước,sau
0,…%2F41752%2Fzol1%2Fï¿½ï¿½ï¿½<br/>pdfï¿½à¼\xadï¿½ï¿½.exe,…%2F41752%2Fzol1%2Fï¿½ï¿½ï¿½<br/>pdfï¿½à¼ï¿½ï¿½.exe
1,http://url.222bz.com/down/é\xad%94æ%96¹?ç½®å¤§?@34_47890.exe,http://url.222bz.com/down/é%94æ%96¹?ç½®å¤§?@34_47890.exe
2,…ata/cheditor4/1009/bookmark/ii.php/?rand\x13inboxlightaspxn.1774256418,…ata/cheditor4/1009/bookmark/ii.php/?randinboxlightaspxn.1774256418
3,…data/cheditor4/1502/bookmark/ii.php?rand\x13InboxLightaspxn.1774256418,…data/cheditor4/1502/bookmark/ii.php?randInboxLightaspxn.1774256418
4,http://londonpearl\xaduk.co/2/R.exe,http://londonpearluk.co/2/R.exe


### 4.4 URL bị "làm mờ" (defang)

Báo cáo an ninh mạng hay viết `[.]` thay cho `.` để URL không bấm được. Khôi phục lại để dòng này còn dùng được.

In [ ]:
apply_rule("Bỏ làm mờ [.] -> .", lambda s: s.str.replace("[.]", ".", regex=False))

Bỏ làm mờ [.] -> .: 1 dòng được sửa


,trước,sau
0,http://ladiesfirst-privileges[.]com/656465/d5678h9.exe,http://ladiesfirst-privileges.com/656465/d5678h9.exe


### 4.5 Scheme sai

- `httpss://...`: gõ thừa chữ `s`.
- `http://http://...`, `http://hxxp://...`, `http://ftp://...`: dán đè scheme lên nhau. Giữ scheme **bên trong** vì đó mới là URL thật.

In [ ]:
SCHEME_ALIAS = {"httpss": "https", "hxxp": "http", "hxxps": "https", "htpp": "http"}


def fix_scheme_typo(s):
    return s.str.replace(r"^(httpss|htpp|hxxps?)://",
                         lambda m: SCHEME_ALIAS[m.group(1).lower()] + "://", regex=True, flags=re.I)


def strip_nested_scheme(s):
    return s.str.replace(r"^[A-Za-z]+://(https?|hxxps?|htpp|ftp):/{1,2}",
                         lambda m: SCHEME_ALIAS.get(m.group(1).lower(), m.group(1).lower()) + "://",
                         regex=True, flags=re.I)


apply_rule("Sửa scheme gõ sai (httpss, htpp, hxxp)", fix_scheme_typo)
apply_rule("Gỡ scheme bị lặp (http://http://...)", strip_nested_scheme)

Sửa scheme gõ sai (httpss, htpp, hxxp): 35 dòng được sửa


,trước,sau
0,httpss://sites.google.com/site/lettheshowbegi…,https://sites.google.com/site/lettheshowbegin…
1,httpss://www.googlemerchandisestore.com/,https://www.googlemerchandisestore.com/
2,httpss://www.microsoft.com/,https://www.microsoft.com/


Gỡ scheme bị lặp (http://http://...): 120 dòng được sửa


,trước,sau
0,http://http://www.oilchangeasheville.com/https:…,http://www.oilchangeasheville.com/https:/www2.S…
1,http://http://leticiaaraujo.com.br/VAN-GOG.969/…,http://leticiaaraujo.com.br/VAN-GOG.969/ATUALIZ…
2,http://http://leticiaaraujo.com.br/VAN-GOG.969/…,http://leticiaaraujo.com.br/VAN-GOG.969/ATUALIZ…


### 4.6 Chữ hoa trong scheme và host

Theo chuẩn URL, scheme và host không phân biệt hoa thường nên đưa về chữ thường (`http://TRIANGLESERVICESLTD.COM/...`). Phần path và query **có** phân biệt hoa thường nên giữ nguyên, phần `user:pass@` cũng vậy.

In [ ]:
def lower_scheme_host(s):
    s = s.copy()
    todo = has_upper_scheme_host(s)

    def fix(m):
        user, at, hostport = m.group(2).rpartition("@")
        return m.group(1).lower() + user + at + hostport.lower()

    s[todo] = s[todo].str.replace(r"^([A-Za-z][A-Za-z0-9+.\-]*://)([^/?#]*)", fix, regex=True)
    return s


apply_rule("Chữ thường cho scheme và host", lower_scheme_host)

Chữ thường cho scheme và host: 160 dòng được sửa


,trước,sau
0,http://bestchaDge.ru,http://bestchadge.ru
1,http://secure1.store.apple.com.gP9MuQlru0z.robstephens.com/sign_in/...,http://secure1.store.apple.com.gp9muqlru0z.robstephens.com/sign_in/...
2,http://TRIANGLESERVICESLTD.COM/25/144.exe,http://triangleservicesltd.com/25/144.exe


## 5. Loại bỏ những dòng không dùng được

Có những dòng không thể sửa một cách chắc chắn. Mỗi lần loại, hàm `drop_rows` lưu URL gốc, nhãn và lý do vào danh sách `dropped` (xuất ra file nhật ký ở mục 7) và ghi lại số dòng còn lại sau mỗi bước.

In [ ]:
dropped = []                                  # các bảng nhỏ: row_id, reason, url_original, type
steps = [("Dữ liệu gốc", 0, len(df))]


def drop_rows(mask, reason):
    global df
    mask = mask.astype(bool)
    ids = df.index[mask]
    dropped.append(pd.DataFrame({"row_id": ids, "reason": reason,
                                 "url_original": raw.loc[ids, "url"].values,
                                 "type": raw.loc[ids, "type"].values}))
    df = df[~mask]
    steps.append((reason, len(ids), len(df)))
    print(f"{reason}: loại {len(ids):,} dòng, còn {len(df):,}")

### 5.1 URL bị cắt cụt

Có những URL kết thúc bằng `...`, tất cả đều là phishing. Gần một nửa số đó dài đúng 73 ký tự, tức 70 ký tự đầu cộng `...`: nhiều khả năng nguồn dữ liệu đã cắt chuỗi khi hiển thị. Hai vấn đề:

1. URL không đầy đủ, không biết phần đuôi, một số dòng còn cụt ngay giữa tên miền.
2. Dấu `...` và độ dài 70–73 trở thành "dấu vết" để mô hình đoán nhãn mà không cần nhìn nội dung URL.

Mặc định ta **loại** các dòng này. Muốn giữ lại thì đặt `DROP_TRUNCATED = False` ở ô cấu hình đầu notebook (khi đó chỉ bỏ dấu `...`).

In [ ]:
truncated = df["url"].str.endswith("...")
print("URL kết thúc bằng '...':", int(truncated.sum()), "| nhãn:", df.loc[truncated, "type"].value_counts().to_dict())
print("Độ dài phổ biến nhất của chúng:", df.loc[truncated, "url"].str.len().value_counts().head(3).to_dict())

if DROP_TRUNCATED:
    drop_rows(truncated, "URL bị cắt cụt (kết thúc bằng '...')")
else:
    apply_rule("Bỏ '...' ở cuối URL bị cắt cụt", lambda s: s.str.replace(r"\.\.\.$", "", regex=True))

URL kết thúc bằng '...': 2555 | nhãn: {'phishing': 2555}
Độ dài phổ biến nhất của chúng: {73: 1226, 63: 93, 49: 77}
URL bị cắt cụt (kết thúc bằng '...'): loại 2,555 dòng, còn 447,621


### 5.2 Host không hợp lệ

Sau khi sửa vẫn còn những dòng không phải URL thật: `http://#NAME?` (trông như lỗi công thức của Excel), `http://php?win=1`, đường dẫn mạng nội bộ `http://\\94.185.85.122\public\xv.exe`, tên máy không có đuôi như `http://netsync1/post.php`...

Host hợp lệ ở đây là các nhãn chữ/số/gạch nối/gạch dưới nối nhau bằng dấu chấm, có ít nhất một dấu chấm (địa chỉ IP cũng thỏa). Dòng nào không thỏa bị loại, bảng dưới liệt kê mẫu để bạn kiểm tra.

In [ ]:
bad_host = ~valid_host(df["url"])
bad_ids = df.index[bad_host]
drop_rows(bad_host, "Host không hợp lệ")
display(raw.loc[bad_ids, ["type", "url"]].assign(url=lambda d: d["url"].str.slice(0, 100)).head(15))

Host không hợp lệ: loại 60 dòng, còn 447,561


,type,url
row_id,,
385991,phishing,http://#NAME?
388981,phishing,http://img851/2304/bismillahus.jpg
388984,phishing,http://micro/advice.php?
389058,phishing,http://user_id=%.4u&version_id=%lu&socks=%lu&build=%lu&crc=%.8x
389060,phishing,http://api/client/pings.php?
389091,phishing,http://\/deutsche_telekom_ag$
389092,phishing,http://\/telekom\/(telekom_deutschland_gmbh|vodafone_onlinerechnung)$
389343,phishing,http://?
389558,phishing,http://url\nreferrer


### 5.3 Trùng lặp

Mỗi URL trong file khác nhau từng ký tự, nhưng nhiều dòng thực chất là **cùng một trang**, chỉ khác cách viết: `http://10000eth.net`, `https://10000eth.net/`, `http://10000eth.net/`... Nếu cùng một trang vừa nằm trong tập huấn luyện vừa nằm trong tập kiểm tra, điểm của mô hình dễ cao một cách giả tạo.

Cách làm: tạo **khóa so sánh** = host (bỏ `www.`) + phần còn lại của URL. Khóa bỏ qua scheme (`http`/`https`) và dấu `/` đứng một mình ở cuối. Path và query vẫn phân biệt hoa thường.

- Cùng khóa nhưng **khác nhãn**: mâu thuẫn, không biết tin nhãn nào, loại cả nhóm.
- Cùng khóa **cùng nhãn**: giữ dòng xuất hiện đầu tiên.

In [ ]:
parts = split_url(df["url"])
rest = parts["rest"].where(parts["rest"] != "/", "")
key = parts["authority"].str.lower().str.replace(r"^www\.", "", regex=True) + rest

conflict = df["type"].groupby(key).transform("nunique") > 1
print("Các dòng cùng trang nhưng khác nhãn:")
display(df.loc[conflict, ["type", "url"]].assign(key=key[conflict]).sort_values("key")[["type", "url"]])
drop_rows(conflict, "Cùng một trang nhưng khác nhãn")

key = key.loc[df.index]
duplicate = key.duplicated(keep="first")
drop_rows(duplicate, "Trùng lặp (cùng trang, cùng nhãn)")

Các dòng cùng trang nhưng khác nhãn:


,type,url
row_id,,
609,legitimate,https://www.4shared.com
395036,phishing,http://4shared.com
206053,legitimate,https://www.google.com/webhp
389066,phishing,http://www.google.com/webhp
107110,legitimate,https://www.tommyhumphreys.com/
399907,phishing,http://tommyhumphreys.com/


Cùng một trang nhưng khác nhãn: loại 6 dòng, còn 447,555
Trùng lặp (cùng trang, cùng nhãn): loại 7,329 dòng, còn 440,226


## 6. Kiểm tra sau khi làm sạch

In [ ]:
compare = pd.DataFrame({"trước": quality_report(raw["url"]), "sau": quality_report(df["url"])})
compare

,trước,sau
HTML entity (&amp; &lt; &#39; ...),5524,0
Có khoảng trắng,521,0
Có ký tự điều khiển / vô hình,5,0
Dạng làm mờ [.],1,0
"Scheme gõ sai (httpss, htpp, hxxp)",35,0
Scheme bị lặp (http://http://...),120,0
Scheme/host viết hoa,160,0
URL bị cắt cụt (kết thúc bằng '...'),2555,0
Host không hợp lệ,658,0


In [ ]:
balance = pd.DataFrame({"trước": raw["type"].value_counts(), "sau": df["type"].value_counts()})
balance["% trước"] = (balance["trước"] / balance["trước"].sum() * 100).round(1)
balance["% sau"] = (balance["sau"] / balance["sau"].sum() * 100).round(1)
display(balance)

summary = pd.DataFrame(steps, columns=["bước", "dòng bị loại", "dòng còn lại"])
display(summary)
print(f"Tổng: {len(raw):,} -> {len(df):,} dòng (giữ {len(df) / len(raw):.1%}).")

,trước,sau,% trước,% sau
type,,,,
legitimate,345738,345605,76.8,78.5
phishing,104438,94621,23.2,21.5


,bước,dòng bị loại,dòng còn lại
0,Dữ liệu gốc,0,450176
1,URL bị cắt cụt (kết thúc bằng '...'),2555,447621
2,Host không hợp lệ,60,447561
3,Cùng một trang nhưng khác nhãn,6,447555
4,"Trùng lặp (cùng trang, cùng nhãn)",7329,440226


Tổng: 450,176 -> 440,226 dòng (giữ 97.8%).


In [ ]:
assert df["url"].str.strip().ne("").all(), "còn URL rỗng"
assert set(df["type"]) <= LABELS, "còn nhãn lạ"
assert not df["url"].duplicated().any(), "còn URL trùng chữ"
assert (compare["sau"] == 0).all(), "còn lỗi chưa xử lý, xem bảng ở trên"
print("Các kiểm tra đều đạt.")

Các kiểm tra đều đạt.


## 7. Lưu kết quả

Ba file được ghi ra:

- `url_dataset_clean.csv`: file sạch, cùng hai cột `url`, `type` như file gốc nên dùng thay thế được ngay.
- `url_dataset_changes.csv`: mỗi dòng đã bị sửa, kèm URL gốc, URL mới và các quy tắc đã áp dụng.
- `url_dataset_dropped.csv`: mỗi dòng đã bị loại, kèm URL gốc, nhãn và lý do.

In [ ]:
clean = df[["url", "type"]]
clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

was_changed = (clean["url"] != raw.loc[clean.index, "url"]).to_numpy()
changed_ids = clean.index[was_changed]
changes = pd.DataFrame({
    "row_id": changed_ids,
    "rules": [" | ".join(applied[i]) for i in changed_ids],
    "url_original": raw.loc[changed_ids, "url"].to_numpy(),
    "url_clean": clean.loc[changed_ids, "url"].to_numpy(),
})
changes.to_csv(CHANGES_PATH, index=False, encoding="utf-8-sig")   # utf-8-sig để Excel đọc đúng tiếng Việt

dropped_df = pd.concat(dropped, ignore_index=True)
dropped_df.to_csv(DROPPED_PATH, index=False, encoding="utf-8-sig")

print(f"{OUTPUT_PATH}: {len(clean):,} dòng")
print(f"{CHANGES_PATH}: {len(changes):,} dòng đã sửa (còn trong file sạch)")
print(f"{DROPPED_PATH}: {len(dropped_df):,} dòng đã loại")
display(dropped_df["reason"].value_counts().to_frame("số dòng"))

url_dataset_clean.csv: 440,226 dòng
url_dataset_changes.csv: 6,116 dòng đã sửa (còn trong file sạch)
url_dataset_dropped.csv: 9,950 dòng đã loại


,số dòng
reason,
"Trùng lặp (cùng trang, cùng nhãn)",7329
URL bị cắt cụt (kết thúc bằng '...'),2555
Host không hợp lệ,60
Cùng một trang nhưng khác nhãn,6


## 8. Lưu ý khi dùng dữ liệu này để huấn luyện mô hình

Phần này không phải làm sạch, nhưng ảnh hưởng trực tiếp đến kết quả, nên cần biết trước.

**Scheme và `www.` làm lộ nhãn.** Gần như mọi URL `legitimate` đều có dạng `https://www.<tên miền>`, kể cả những host trông như được gắn tự động (ví dụ `www.en.wikipedia.org`). Ngược lại phishing chủ yếu là `http://` không `www.`. Nhiều khả năng đây là dấu vết của cách dựng bộ dữ liệu chứ không phải đặc điểm của trang web.

In [ ]:
host = get_host(clean["url"])
has_www = host.str.startswith("www.")
is_https = clean["url"].str.startswith("https://")

share = pd.DataFrame({"tỉ lệ có www.": has_www.groupby(clean["type"]).mean(),
                      "tỉ lệ dùng https": is_https.groupby(clean["type"]).mean()}).round(3)
display(share)

guess = has_www.map({True: "legitimate", False: "phishing"})
print(f"Chỉ cần đoán 'có www. thì là legitimate' đã đúng {(guess == clean['type']).mean():.1%} số dòng.")

,tỉ lệ có www.,tỉ lệ dùng https
type,,
legitimate,0.998,1.000
phishing,0.072,0.063


Chỉ cần đoán 'có www. thì là legitimate' đã đúng 98.3% số dòng.


Nếu mục tiêu là phân loại phishing dựa trên nội dung URL, nên bỏ scheme và `www.` trước khi trích đặc trưng. Cột `url_core` dưới đây làm việc đó (không lưu vào file sạch, để file sạch giữ nguyên cấu trúc gốc):

In [ ]:
url_core = (clean["url"]
            .str.replace(r"^[a-z]+://", "", regex=True)
            .str.replace(r"^www\.", "", regex=True))
clean.assign(url_core=url_core).sample(5, random_state=0)

,url,type,url_core
row_id,,,
198265,https://www.flickr.com/photos/walkingsf/5448636051/,legitimate,flickr.com/photos/walkingsf/5448636051/
148926,https://www.bellator.com/ViewArticle.dbml?DB_OEM_ID=23600&ATCLID=204955118,legitimate,bellator.com/ViewArticle.dbml?DB_OEM_ID=23600&ATCLID=204955118
161810,https://www.chargers.com/schedule/index.html,legitimate,chargers.com/schedule/index.html
388650,http://darknightpress.org/yahoo-inc.com/fxage.htm,phishing,darknightpress.org/yahoo-inc.com/fxage.htm
115056,https://www.waatp.com/people/james-r-bajorek/41081604/,legitimate,waatp.com/people/james-r-bajorek/41081604/


Hai điểm nữa:

- **Nhãn lệch** (tỉ lệ ở bảng mục 6): khi chia train/test nên chia phân tầng (*stratified*) và đánh giá bằng precision, recall, F1 của lớp `phishing`, đừng chỉ nhìn accuracy.
- **Một số tên miền chiếm rất nhiều dòng** (bảng dưới). Nếu chia ngẫu nhiên theo dòng, cùng một tên miền sẽ nằm ở cả hai phía train và test; cân nhắc chia theo tên miền.

In [ ]:
top_hosts = host.value_counts().head(5)
pd.DataFrame({"số dòng": top_hosts, "% tổng": (top_hosts / len(clean) * 100).round(2)})

,số dòng,% tổng
host,,
www.en.wikipedia.org,12668,2.88
www.youtube.com,8618,1.96
www.facebook.com,7741,1.76
www.amazon.com,4618,1.05
www.imdb.com,3470,0.79


In [ ]:
import pandas as pd
import re
from google.colab import files

# 1. Đọc dữ liệu sạch
df = pd.read_csv('url_dataset_clean.csv')

# 2. Trích xuất các đặc trưng cấu trúc đường link (Lexical features)
df['url_len'] = df['url'].apply(lambda i: len(str(i)))
df['count_dot'] = df['url'].apply(lambda i: str(i).count('.'))
df['count_hyphen'] = df['url'].apply(lambda i: str(i).count('-'))
df['count_slash'] = df['url'].apply(lambda i: str(i).count('/'))
df['count_question'] = df['url'].apply(lambda i: str(i).count('?'))
df['count_equal'] = df['url'].apply(lambda i: str(i).count('='))
df['count_at'] = df['url'].apply(lambda i: str(i).count('@'))
df['count_digits'] = df['url'].apply(lambda i: sum(c.isdigit() for c in str(i)))
df['has_ip'] = df['url'].apply(lambda i: 1 if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', str(i)) else 0)

# 3. Sắp xếp lại thứ tự cột
feature_cols = ['url_len', 'count_dot', 'count_hyphen', 'count_slash',
                'count_question', 'count_equal', 'count_at', 'count_digits', 'has_ip', 'type', 'url']
df = df[feature_cols]

# 4. Lưu ra file mới và tải tự động về máy tính
df.to_csv('url_features.csv', index=False)
print("Trích xuất đặc trưng hoàn tất! Đang tải file url_features.csv về máy...")
files.download('url_features.csv')

Trích xuất đặc trưng hoàn tất! Đang tải file url_features.csv về máy...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>